# Занятие 28. Практика: логистическая регрессия и завершение курса

Это второе занятие по теме логистической регрессии: практическая работа в формате учебного рейтинга решений. Команды по 1–2 человека строят модель, сохраняют `submission.csv`, а преподаватель считает результат на скрытой test-выборке.

В блокноте уже есть рабочий стартовый baseline. Сначала запустите его и убедитесь, что получается `submission.csv`, затем улучшайте признаки, регуляризацию и порог.

**Задача:** предсказать, завершит ли ученик курс. Целевой столбец в train — `will_finish`.

**Главная метрика занятия:** F1-мера (F1-score). Больше — лучше.

F1 полезна здесь потому, что нам важны и найденные будущие “завершившие курс”, и количество ложных срабатываний.


## Правила и ориентир на 90 минут

1. 0–10 мин — понять данные, целевой класс и метрику.
2. 10–30 мин — запустить стартовый baseline логистической регрессии.
3. 30–60 мин — улучшить признаки и регуляризацию.
4. 60–80 мин — подобрать порог вероятности для F1.
5. 80–90 мин — сохранить submission и обсудить ошибки.

Используйте `test.csv` только для финального прогноза. Ответы к test не должны участвовать в выборе модели.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_DIR = Path("competition") / "data"


In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print(train.shape, test.shape)
display(train.head())
display(sample_submission.head())


## Validation и стратификация

Стратификация нужна, чтобы доля классов 0 и 1 была похожей в train и validation.


In [ ]:
target = "will_finish"
id_col = "id"

features = [c for c in train.columns if c not in [target, id_col]]
X = train[features]
y = train[target]

print("Доля класса 1:", y.mean().round(3))

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)


In [ ]:
def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

numeric_preprocess = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_preprocess = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_ohe()),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocess, numeric_features),
        ("cat", categorical_preprocess, categorical_features),
    ]
)


## Первый baseline: вероятность → класс

Логистическая регрессия выдаёт вероятность класса 1. Чтобы получить класс, выбираем порог.

Если порог 0.5, то:

- вероятность 0.49 даёт класс 0;
- вероятность 0.51 даёт класс 1.


In [ ]:
model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", LogisticRegression(max_iter=1000)),
    ]
)

model.fit(X_train, y_train)
val_proba = model.predict_proba(X_val)[:, 1]
val_pred = (val_proba >= 0.5).astype(int)

print("accuracy:", round(accuracy_score(y_val, val_pred), 3))
print("precision:", round(precision_score(y_val, val_pred), 3))
print("recall:", round(recall_score(y_val, val_pred), 3))
print("F1:", round(f1_score(y_val, val_pred), 3))
print("ROC-AUC:", round(roc_auc_score(y_val, val_proba), 3))


## Подбор порога

Модель может хорошо расставлять вероятности, но порог 0.5 не всегда лучший для F1.

Ниже мы перебираем пороги и выбираем тот, где F1 на validation максимальна.


In [ ]:
thresholds = np.linspace(0.15, 0.85, 71)
rows = []

for threshold in thresholds:
    pred = (val_proba >= threshold).astype(int)
    rows.append(
        {
            "threshold": threshold,
            "precision": precision_score(y_val, pred, zero_division=0),
            "recall": recall_score(y_val, pred, zero_division=0),
            "f1": f1_score(y_val, pred, zero_division=0),
        }
    )

threshold_report = pd.DataFrame(rows).sort_values("f1", ascending=False)
threshold_report.head(10)


In [ ]:
best_threshold = float(threshold_report.iloc[0]["threshold"])
print("Лучший порог на validation:", round(best_threshold, 3))


## Идеи для улучшения

Попробуйте несколько вариантов и сравнивайте их на validation:

- `LogisticRegression(C=0.3)`, `C=1`, `C=3`;
- `class_weight="balanced"`;
- новые признаки:
  - `activity_total` — общий учебный ритм: часы практики + часть посещений клуба;
  - `score_per_missed_class` — последняя оценка с поправкой на число пропусков;
- проверить удаление слабого или дублирующего признака: убрать его, переобучить модель и сравнить F1 на той же validation-выборке;
- другой порог вероятности.

Не подбирайте порог по test: там нет честных ответов.


In [ ]:
def add_features(df):
    df = df.copy()
    df["activity_total"] = df["practice_hours_week"] + 0.5 * df["club_visits"]
    df["score_per_missed_class"] = df["last_score"] / (df["missed_classes"] + 1)
    df["teacher_messages_per_week"] = df["messages_to_teacher"] / 4
    return df


train_fe = add_features(train)
test_fe = add_features(test)

features_fe = [c for c in train_fe.columns if c not in [target, id_col]]
X = train_fe[features_fe]
y = train_fe[target]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

preprocess_fe = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocess, numeric_features),
        ("cat", categorical_preprocess, categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocess", preprocess_fe),
        ("model", LogisticRegression(max_iter=1000, C=1.0, class_weight=None)),
    ]
)

model.fit(X_train, y_train)
val_proba = model.predict_proba(X_val)[:, 1]

thresholds = np.linspace(0.15, 0.85, 71)
best_threshold = max(
    thresholds,
    key=lambda t: f1_score(y_val, (val_proba >= t).astype(int), zero_division=0),
)
val_pred = (val_proba >= best_threshold).astype(int)

print("threshold:", round(float(best_threshold), 3))
print("precision:", round(precision_score(y_val, val_pred, zero_division=0), 3))
print("recall:", round(recall_score(y_val, val_pred, zero_division=0), 3))
print("F1:", round(f1_score(y_val, val_pred, zero_division=0), 3))
print("ROC-AUC:", round(roc_auc_score(y_val, val_proba), 3))


## Финальное обучение и submission

После выбора модели обучаем её на всём `train.csv` и делаем прогноз для `test.csv`.


In [ ]:
final_X = train_fe[features_fe]
final_y = train_fe[target]

final_model = model
final_model.fit(final_X, final_y)

test_proba = final_model.predict_proba(test_fe[features_fe])[:, 1]
test_pred = (test_proba >= best_threshold).astype(int)

submission = pd.DataFrame(
    {
        "id": test_fe["id"],
        "probability": np.round(test_proba, 5),
        "will_finish": test_pred,
    }
)

submission.to_csv("submission.csv", index=False)
submission.head()


## Что сдать

Сдайте файл `submission.csv`.

В нём должны быть столбцы:

- `id`;
- `probability` — вероятность класса 1;
- `will_finish` — итоговый прогноз 0 или 1.

Главная метрика считается по `will_finish`.
